### Deploying OHIF Viewer in a Serverless Lakehouse App
This notebook guides you through the process of deploying the OHIF Viewer as a serverless lakehouse application.

In [0]:
%pip install --upgrade databricks-sdk==0.88.0 psycopg[binary,pool] psycopg2-binary fsspec -q
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

# Initializing Environment and Setting Up Application

Initialize widgets to capture the SQL warehouse ID, table, and volume. We also set up the environment and define the application name as "pixels-ohif-viewer".

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [0]:
app_name = "pixels-dicomweb"
app_gateway_name = app_name + "-gateway"
lakebase_instance_name = "pixels-lakebase"
serving_endpoint_name = "pixels-monai-uc"

# Setting Up and Deploying the Lakehouse Application

The next step will perform several critical steps to set up and deploy our Lakehouse Application:

1. **Import Necessary Libraries**: We start by importing required libraries and modules such as `AppResource`, `AppResourceSqlWarehouse`, and others from the `databricks.sdk.service.apps`, along with `Path` from `pathlib`, and `dbx.pixels.resources`.

 2. **Initialize Workspace Client**: An instance of `WorkspaceClient` is created to interact with the Databricks workspace.

 3. **Prepare Application Configuration**: The application's configuration is prepared by reading a template configuration file (`app-config.yaml`), replacing placeholders with actual values (like the pixels table name), and writing the modified configuration to `app.yaml`.

 4. **Define SQL Warehouse Resource**: We define a `sql_resource` with the SQL warehouse ID and permissions required for the application to use the SQL warehouse.

 5. **Create and Deploy the Application**: The application is created and deployed using the `create_and_wait` and `deploy_and_wait` methods of the `WorkspaceClient`. This process involves specifying the application name, resources (like the SQL warehouse resource), and the path to the application's source code.

 6. **Extract Service Principal ID**: After deployment, the service principal ID is extracted from the deployment artifacts for permission grants.

 7. **Output Deployment Status and URL**: Finally, the deployment status message and the application URL are printed, indicating the completion of the deployment process and how to access the deployed application.

 This cell encapsulates the entire process of preparing, creating, and deploying the Lakehouse Application, making it a pivotal step in the application setup workflow.

# Create Lakebase DB and DICOM_FRAMES table

In [0]:
import dbx
from dbx.pixels.lakebase import LakebaseUtils

lb_utils = LakebaseUtils(instance_name=lakebase_instance_name, uc_table_name=table, create_instance=True, min_cu=0.5, max_cu=2.0)

path = os.path.dirname(dbx.pixels.__file__)
sql_base_path = f"{path}/resources/sql/lakebase"

# Database is aligned to UC catalog, schema to UC schema
# (e.g. catalog.schema.table → database=catalog, schema=schema)
_lb_schema = lb_utils.schema

for sql_file in ["CREATE_LAKEBASE_SCHEMA.sql", "CREATE_LAKEBASE_DICOM_FRAMES.sql", "CREATE_LAKEBASE_METRICS.sql"]:
    file_path = os.path.join(sql_base_path, sql_file)
    with open(file_path, "r") as file:
        lb_utils.execute_query(file.read().format(schema_name=_lb_schema))

# Create the UC view used by Reverse ETL to sync instance_paths into Lakebase
from dbx.pixels.lakebase import parse_uc_table_name
_uc_catalog, _uc_schema, _uc_table = parse_uc_table_name(table)
with open(os.path.join(sql_base_path, "CREATE_INSTANCE_PATHS_VIEW.sql"), "r") as file:
    spark.sql(file.read().format(catalog=_uc_catalog, schema=_uc_schema, table=_uc_table))
print(f"✓ Created UC view {_uc_catalog}.{_uc_schema}.instance_paths_vw for Reverse ETL Sync")

# Create Reverse ETL Synced Table

Sync the `instance_paths_vw` view from Unity Catalog into Lakebase using
Reverse ETL.  This creates:

1. A **read-only synced table** in UC (`catalog.schema.instance_paths`)
2. A **Postgres table** in Lakebase (`schema.instance_paths`)

The sync pipeline keeps the Lakebase table updated so the
DICOMweb app can resolve SOP Instance UIDs → file paths in sub-10 ms
without querying the SQL warehouse.

In [ ]:
import json, time

_synced_table_name = f"{_uc_catalog}.{_uc_schema}.instance_paths"

# Get project and branch UIDs from the Lakebase postgres project
_project = lb_utils.workspace_client.postgres.get_project(lb_utils.project_resource_name)
_branch = lb_utils.workspace_client.postgres.get_branch(lb_utils.branch_resource_name)

# Check if synced table already exists in Unity Catalog
_needs_create = True
try:
    _tbl = w.tables.get(_synced_table_name)
    if _tbl.table_type and _tbl.table_type.value == "FOREIGN":
        print(f"✓ Synced table '{_synced_table_name}' already exists (pipeline_id={_tbl.pipeline_id})")
        _needs_create = False
    else:
        print(f"⚠ Table '{_synced_table_name}' exists but is not a synced table (type={_tbl.table_type})")
        _needs_create = False
except Exception:
    pass  # Table doesn't exist in UC — need to create it

if _needs_create:
    # Drop any stale Postgres table left from a previous failed sync
    try:
        lb_utils.execute_query(f"DROP TABLE IF EXISTS {_lb_schema}.instance_paths")
    except Exception:
        pass  # Table may not exist — that's fine

    _payload = {
        "name": _synced_table_name,
        "database_project_id": _project.uid,
        "database_branch_id": _branch.uid,
        "logical_database_name": _uc_catalog,
        "spec": {
            "source_table_full_name": f"{_uc_catalog}.{_uc_schema}.instance_paths_vw",
            "primary_key_columns": ["local_path"],
            "scheduling_policy": "SNAPSHOT",
        },
    }

    try:
        _result = w.api_client.do("POST", "/api/2.0/database/synced_tables", body=_payload)
        _pipeline_id = _result.get("data_synchronization_status", {}).get("pipeline_id", "unknown")
        print(f"✓ Synced table created — pipeline {_pipeline_id} running")

        # Poll until online or failed (max ~2 min)
        for _i in range(12):
            time.sleep(10)
            _status_resp = w.api_client.do("GET", f"/api/2.0/database/synced_tables/{_synced_table_name}")
            _state = _status_resp.get("data_synchronization_status", {}).get("detailed_state", "")
            if "ONLINE" in _state:
                print(f"✓ Synced table is online")
                break
            elif "FAIL" in _state:
                _msg = _status_resp.get("data_synchronization_status", {}).get("message", "")
                print(f"✗ Synced table failed: {_msg}")
                break
    except Exception as _e:
        print(f"✗ Failed to create synced table: {_e}")

# Deploy DICOMWeb App Gateway
Full DICOMweb server (QIDO-RS, WADO-RS, WADO-URI, STOW-RS) backed by Databricks SQL, Lakebase, and Volumes

In [0]:
from databricks.sdk.service.apps import (
    AppResource,
    AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission,
    App,
    AppDeployment,
    ComputeSize
)
from pathlib import Path
import os

In [ ]:
import shutil, os, glob

# Compute repo root from notebook path
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_gateway_path = os.path.join(_repo_root, "apps", "dicom-web-gateway")
_dicomweb_path = os.path.join(_repo_root, "apps", "dicom-web")
_view_app_path = os.path.join(_repo_root, "apps", "view-app")

# Derive STOW volume path from the volume widget (catalog.schema.volume_name)
_vol_parts = volume.split(".")
_stow_volume_path = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}/stow/"

# Copy the pre-built wheel into each app directory so pip can install it locally
_wheel_pattern = os.path.join(_repo_root, "dist", "databricks_pixels-*.whl")
_wheels = sorted(glob.glob(_wheel_pattern))
assert _wheels, f"No wheel found at {_wheel_pattern} — run 'make build' before deploying"
_wheel_path = _wheels[-1]  # latest version

for _app_dir in [_dicomweb_path, _dicomweb_gateway_path, _view_app_path]:
    _dest = os.path.join(_app_dir, os.path.basename(_wheel_path))
    shutil.copy2(_wheel_path, _dest)
print(f"Copied {os.path.basename(_wheel_path)} into app directories")

# Generate app.yml from the template
with open(f"{_dicomweb_gateway_path}/app-config.yml", "r") as config_input:
    with open(f"{_dicomweb_gateway_path}/app.yml", "w") as config_output:
        config_output.write(
            config_input.read()
            .replace("{PIXELS_TABLE}", table)
            .replace("{LAKEBASE_INSTANCE_NAME}", lakebase_instance_name)
            .replace("{STOW_VOLUME_PATH}", _stow_volume_path)
        )

In [ ]:
sql_resource = AppResource(
    name="sql_warehouse",
    sql_warehouse=AppResourceSqlWarehouse(
        id=sql_warehouse_id,
        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
    ),
)

# Create or retrieve existing app
if app_gateway_name in [a.name for a in w.apps.list()]:
    print(f"App '{app_gateway_name}' already exists — updating resources")
    app = w.apps.update(app_gateway_name, App(app_gateway_name, resources=[sql_resource]))
else:
    print(f"Creating DICOMweb App GATEWAY '{app_gateway_name}' — this may take a few minutes …")
    app = App(
        app_gateway_name,
        default_source_code_path=_dicomweb_gateway_path,
        user_api_scopes=["sql", "files.files"],
        resources=[sql_resource],
        compute_size=ComputeSize.LARGE
    )
    app = w.apps.create_and_wait(app)
    print(f"✓ App created: {app.url}")

# Deploy DICOMweb OHIF App

Serves the OHIF viewer, MONAI, redaction and VLM APIs; reverse-proxies all DICOMweb protocol traffic to the gateway

In [ ]:
# Generate app.yml from the template
with open(f"{_dicomweb_path}/app-config.yml", "r") as config_input:
    with open(f"{_dicomweb_path}/app.yml", "w") as config_output:
        config_output.write(
            config_input.read()
            .replace("{PIXELS_TABLE}", table)
            .replace("{DICOMWEB_GATEWAY_URL}", app.url)
        )

print(f"Generated app.yml for DICOMweb app")

In [ ]:
resources = [sql_resource]

if serving_endpoint_name in [ep.name for ep in w.serving_endpoints.list()]:
    resources.append(
        AppResource(
            name="serving_endpoint",
            serving_endpoint=AppResourceServingEndpoint(
                name=serving_endpoint_name,
                permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY,
            ),
        )
    )

# Create or retrieve existing app
if app_name in [a.name for a in w.apps.list()]:
    print(f"App '{app_name}' already exists — updating resources")
    app = w.apps.update(app_name, App(app_name, resources=resources))
else:
    print(f"Creating DICOMweb App '{app_name}' — this may take a few minutes …")
    app = App(
        app_name,
        default_source_code_path=_dicomweb_path,
        user_api_scopes=["sql", "files.files"],
        resources=resources,
    )
    app = w.apps.create_and_wait(app)
    print(f"✓ App created: {app.url}")

# Granting Permissions

Grant the DICOMweb app's service principal access to:
- Lakebase tables (`dicom_frames`, `instance_paths`)
- Unity Catalog (catalog, schema, table, volume)

In [0]:
from databricks.sdk.service import catalog

In [0]:
app_instance = w.apps.get(app_gateway_name)
service_principal_id = app_instance.service_principal_client_id
role = lb_utils.get_or_create_sp_role(service_principal_id)

In [ ]:
# Lakebase grants
lb_utils.execute_query(f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.dicom_frames TO "{service_principal_id}"')

# instance_paths is created by the synced table pipeline — grant only if sync has completed
try:
    lb_utils.execute_query(f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.instance_paths TO "{service_principal_id}"')
except Exception as e:
    if "does not exist" in str(e):
        print(f"⚠ Skipping instance_paths grant — table not yet synced (pipeline may still be running)")
    else:
        raise

lb_utils.execute_query(f'GRANT SELECT, INSERT, UPDATE, DELETE ON {_lb_schema}.endpoint_metrics TO "{service_principal_id}"')
lb_utils.execute_query(f'GRANT USAGE ON SCHEMA {_lb_schema} TO "{service_principal_id}"')
lb_utils.execute_query(f'ALTER DEFAULT PRIVILEGES IN SCHEMA {_lb_schema} GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO "{service_principal_id}"')

In [ ]:
# Index for fast Study Instance UID lookups on the synced table
try:
    lb_utils.execute_query(f'CREATE INDEX IF NOT EXISTS idx_instance_paths_study ON {_lb_schema}.instance_paths (study_instance_uid)')
except Exception as e:
    if "does not exist" in str(e):
        print(f"⚠ Skipping instance_paths index — table not yet synced (pipeline may still be running)")
    else:
        raise

In [0]:
# UC grants
w.grants.update(
    full_name=_uc_catalog,
    securable_type="catalog",
    changes=[catalog.PermissionsChange(add=[catalog.Privilege.USE_CATALOG], principal=service_principal_id)],
)
w.grants.update(
    full_name=f"{_uc_catalog}.{_uc_schema}",
    securable_type="schema",
    changes=[catalog.PermissionsChange(add=[catalog.Privilege.USE_SCHEMA], principal=service_principal_id)],
)
w.grants.update(
    full_name=table,
    securable_type="table",
    changes=[catalog.PermissionsChange(add=[catalog.Privilege.ALL_PRIVILEGES], principal=service_principal_id)],
)
w.grants.update(
    full_name=volume,
    securable_type="volume",
    changes=[catalog.PermissionsChange(add=[catalog.Privilege.ALL_PRIVILEGES], principal=service_principal_id)],
)

In [0]:
print("✓ Permissions granted")

# Deploy

In [ ]:
app_gateway_deploy = w.apps.deploy_and_wait(app_gateway_name, AppDeployment(source_code_path=_dicomweb_gateway_path))
app_deploy = w.apps.deploy_and_wait(app_name, AppDeployment(source_code_path=_dicomweb_path))

In [0]:
print(f"✓ {app_deploy.status.message}")
print(f"  URL: {app.url}")